In [1]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['pecanproject_pecan',	'esa-philab_opensartoolkit', 'stolllab_easyspin',	'mariohsouto_proxsdp.jl',
'qiskit_qiskit-experiments',	'ericproffitt_topicmodelsvb.jl',	'vcityteam_ud-reproducibility',
'hbctraining_scrna-seq_online',	'smileipic_smilei',	'geofranzi_geoportal']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,00002128a7de94e5e9d1df1e9fcacff8183bcdad,pecanproject_pecan


In [ ]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  1%|          | 63/7162 [01:08<2:08:27,  1.09s/it]

Got Errors {'0440008edc4bf973a152ccdd8c6fc46415f2bb17': 'Key 0440008edc4bf973a152ccdd8c6fc46415f2bb17 not found in /da5_fast/All.sha1c/commit_4.tch'}


  1%|          | 88/7162 [01:36<2:07:38,  1.08s/it]

Got Errors {'05f1f47f3e5139c750ae05689ec91ac4d441ca98': 'Key 05f1f47f3e5139c750ae05689ec91ac4d441ca98 not found in /da5_fast/All.sha1c/commit_5.tch'}


  2%|▏         | 123/7162 [02:14<2:09:00,  1.10s/it]

...

Got Errors {'c1df0683c3be5c577c490a19d4f785c6daec309f': 'Key c1df0683c3be5c577c490a19d4f785c6daec309f not found in /da5_fast/All.sha1c/commit_65.tch'}


 97%|█████████▋| 6927/7162 [2:05:38<04:21,  1.11s/it]

Got Errors {'e5323ceaa72f7162d4125c7f6f23989c0d090db3': 'Key e5323ceaa72f7162d4125c7f6f23989c0d090db3 not found in /da5_fast/All.sha1c/commit_101.tch'}


100%|██████████| 7162/7162 [2:09:54<00:00,  1.09s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,00002128a7de94e5e9d1df1e9fcacff8183bcdad,c468e14dfbd6f5393d078c2c0439405264813e4c,[d6138d49eb85eb2cf86b8567b6824756578acab0],Betsy Cowdery <emcowdery@gmail.com>,1479233072,-0500,Betsy Cowdery <emcowdery@gmail.com>,1479233072,-0500,Fixing external function call\n,00002128a7de94e5e9d1df1e9fcacff8183bcdad,pecanproject_pecan
1,0000896d56a7024d7019a01bc671771176ed3800,909e2fac51199214661e9922d0949dfa76fdc22a,[ea9862bd3ee531328a34397ee5714d4018a242d6],Betsy Cowdery <emcowdery@gmail.com>,1478089294,-0400,Betsy Cowdery <emcowdery@gmail.com>,1478089307,-0400,Adding optional argument var.ids so that only ...,0000896d56a7024d7019a01bc671771176ed3800,pecanproject_pecan


In [3]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '00002128a7de94e5e9d1df1e9fcacff8183bcdad', 'tree': 'c468e14dfbd6f5393d078c2c0439405264813e4c', 'parent': ['d6138d49eb85eb2cf86b8567b6824756578acab0'], 'author': 'Betsy Cowdery <emcowdery@gmail.com>', 'author_time': 1479233072, 'author_tz': '-0500', 'committer': 'Betsy Cowdery <emcowdery@gmail.com>', 'committer_time': 1479233072, 'committer_tz': '-0500', 'message': 'Fixing external function call\n'}


In [52]:
dfinf = df_commit_data[['project', 'commit', 'author', 'author_time', 'message']].copy()
dfinf.rename(columns={'author_time': 'time'}, inplace=True)

# Sanitize all whitespace, newlines, and semicolons
dfinf['message'] = (
    dfinf['message']
    .astype(str)
    .str.replace(r'[\r\n]+', ' ', regex=True)
    .str.replace(';', ',')
    .str.strip()
)

In [53]:
print("Shape:", dfinf.shape)

Shape: (71531, 5)


In [63]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
#dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
#for k in commit_data:
#  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
#  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [54]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,pecanproject_pecan,00002128a7de94e5e9d1df1e9fcacff8183bcdad,Betsy Cowdery <emcowdery@gmail.com>,1479233072,Fixing external function call


In [64]:
dfinf.shape

(71531, 5)

In [62]:
#mode a means append, so you have all your projects in the same file
yournetid='jbronyah'
#dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)
dfinf.to_csv(yournetid + '_project_summary.csv', index=False, sep=';', mode='w', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

In [61]:
import pandas as pd

# Compute World of Code (WoC) metrics per project from df_commit_data
woc_stats = df_commit_data.groupby('project').agg(
    commits=('commit', 'count'),
    authors=('author', 'nunique'),
    From=('author_time', 'min'),
    To=('author_time', 'max')
).reset_index()

woc_stats.rename(columns={
    'commits': '#commits',
    'authors': '#authors'
}, inplace=True)

# GitHub repository metrics
gh_metrics = pd.DataFrame([
    {
        'project': 'pecanproject_pecan',
        '#stars': 246,
        '#forks': 321,
        'LastGHCommitDate': '2026/09/25'
    },
    {
        'project': 'esa-philab_opensartoolkit',
        '#stars': 248,
        '#forks': 60,
        'LastGHCommitDate': '2026/02/04'
    },
    {
        'project': 'stolllab_easyspin',
        '#stars': 66,
        '#forks': 29,
        'LastGHCommitDate': '2026/09/28'
    },
    {
        'project': 'mariohsouto_proxsdp.jl',
        '#stars': 97,
        '#forks': 13,
        'LastGHCommitDate': '2025/05/25'
    },
    {
        'project': 'qiskit_qiskit-experiments',
        '#stars': 197,
        '#forks': 139,
        'LastGHCommitDate': '2026/09/09'
    },
    {
        'project': 'ericproffitt_topicmodelsvb.jl',
        '#stars': 84,
        '#forks': 9,
        'LastGHCommitDate': '2025/02/09'
    },
    {
        'project': 'vcityteam_ud-reproducibility',
        '#stars': 2,
        '#forks': 2,
        'LastGHCommitDate': '2024/11/21'
    },
    {
        'project': 'hbctraining_scrna-seq_online',
        '#stars': 672,
        '#forks': 209,
        'LastGHCommitDate': '2026/08/20'
    },
    {
        'project': 'smileipic_smilei',
        '#stars': 449,
        '#forks': 157,
        'LastGHCommitDate': '2026/09/14'
    },
    {
        'project': 'geofranzi_geoportal',
        '#stars': 0,
        '#forks': 0,
        'LastGHCommitDate': '2025/05/12'
    },
])

# Merge WoC statistics with GitHub data
df_project_stats = woc_stats.merge(gh_metrics, on='project', how='left')

# 4. Display the table in the notebook
display(df_project_stats)

# Export
#df_project_stats.to_csv('jbronyah_project_stats.csv', sep=';', index=False)

,project,#commits,#authors,From,To,#stars,#forks,LastGHCommitDate
0,ericproffitt_topicmodelsvb.jl,1542,11,1466824003,1741617121,84,9,2025/02/09
1,esa-philab_opensartoolkit,637,31,1542903465,1758635026,248,60,2026/02/04
2,geofranzi_geoportal,1169,14,1467280640,1756391053,0,0,2025/05/12
3,hbctraining_scrna-seq_online,1619,16,1514765279,1744731382,672,209,2026/08/20
4,mariohsouto_proxsdp.jl,831,20,1515506544,1748191005,97,13,2025/05/25
5,pecanproject_pecan,38951,454,1283468974,1776459632,246,321,2026/09/25
6,qiskit_qiskit-experiments,11635,145,1614191172,1762367916,197,139,2026/09/09
7,smileipic_smilei,11931,212,1364546318,1761655373,449,157,2026/09/14
8,stolllab_easyspin,2542,41,1318928290,1759430732,66,29,2026/09/28
9,vcityteam_ud-reproducibility,674,26,1576158853,1732201063,2,2,2024/11/21
